# ਪਾਠ 13 - ਏਜੰਟ ਮੈਮੋਰੀ


## ਸੈਟਅਪ

ਇਹ ਨੋਟਬੁੱਕ ਦਿਖਾਉਂਦਾ ਹੈ ਕਿ ਕਿਵੇਂ **ਮਾਈਕ੍ਰੋਸਾਫਟ ਏਜੰਟ ਫ੍ਰੇਮਵਰਕ** (MAF) ਦੀ ਵਰਤੋਂ ਕਰਕੇ **ਟਿਕਾਊ ਮੇਮੋਰੀ** ਨਾਲ ਯਾਤਰਾ ਬੁਕਿੰਗ ਏਜੰਟ ਬਣਾਇਆ ਜਾ ਸਕਦਾ ਹੈ।

ਤੁਸੀਂ ਸਿੱਖੋਗੇ ਕਿ ਕਿਵੇਂ ਵੱਖ-ਵੱਖ ਕਿਸਮਾਂ ਦੀ ਏਜੰਟ ਮੇਮੋਰੀ — ਵਰਕਿੰਗ, ਛੋਟੇ ਸਮੇਂ ਦੀ, ਅਤੇ ਲੰਮੇ ਸਮੇਂ ਦੀ — ਗੱਲਬਾਤ ਦੌਰਾਨ ਜਾਣਕਾਰੀ ਨੂੰ ਕਿਵੇਂ ਰੱਖਦੀ ਅਤੇ ਵਰਤਦੀ ਹੈ।

**ਤਿਆਰੀਆਵਾਂ:**
- ਇੱਕ Azure AI Foundry ਪ੍ਰੋਜੈਕਟ ਜਿਸ ਨਾਲ ਇੱਕ ਚੈਟ ਮਾਡਲ (ਜਿਵੇਂ `gpt-4o-mini`) ਡਿਪਲੋਇਆ ਹੋਇਆ ਹੋਵੇ।
- Azure CLI ਨਾਲ ਲੌਗਇਨ ਹੋਇਆ ਹੋਵੇ — ਟਰਮੀਨਲ ਵਿੱਚ `az login` ਚਲਾਓ।
- `AZURE_AI_PROJECT_ENDPOINT` — ਤੁਹਾਡੇ Azure AI Foundry ਪ੍ਰੋਜੈਕਟ ਦਾ ਐਂਡਪੋਇੰਟ।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ਤੁਹਾਡੇ ਡਿਪਲੋਇਆ ਮਾਡਲ ਦਾ ਨਾਮ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ਏਜੰਟ ਮੈਮੋਰੀ ਦੇ ਕਿਸਮਾਂ

ਏਆਈ ਏਜੰਟ ਵੱਖ-ਵੱਖ ਕਿਸਮ ਦੀ ਮੈਮੋਰੀ ਦਾ ਲਾਹਾ ਉਠਾ ਸਕਦੇ ਹਨ, ਜੋ ਹਰ ਇੱਕ ਵੱਖਰੇ ਮਕਸਦ ਨੂੰ ਸੇਵਾ ਦਿੰਦੀ ਹੈ:

### ਵਰਕਿੰਗ ਮੈਮੋਰੀ
ਗੱਲਬਾਤ ਦਾ ਧਾਗਾ ਖੁਦ — ਇੱਕਲ Session ਵਿੱਚ ਬਦਲੇ ਗਏ ਸੰਦੇਸ਼। ਏਜੰਟ coherence ਨੂੰ ਬਣਾਈ ਰੱਖਣ ਲਈ ਉਸੇ ਧਾਗੇ ਵਿੱਚ ਪਹਿਲਾਂ ਦੇ ਸੰਦੇਸ਼ਾਂ ਨੂੰ ਵਾਪਸ ਦੇਖ ਸਕਦਾ ਹੈ। MAF ਵਿੱਚ ਇਹ **`agent.create_session()`** ਰਾਹੀਂ ਬਣਾਈ ਜਾਂਦੀ ਹੈ, ਜੋ ਇੱਕ `AgentSession` ਨੂੰ ਵਾਪਸ ਕਰਦੀ ਹੈ।

### ਸ਼ਾਰਟ-ਟਾਰਮ ਮੈਮੋਰੀ
ਜਾਣਕਾਰੀ ਜੋ ਇਕ ਕਾਰਜ ਜਾਂ ਸੈਸ਼ਨ ਦੀ ਮਿਆਦ ਤਕ ਟਿਕਦੀ ਹੈ ਪਰ ਸਥਾਈ ਤੌਰ 'ਤੇ ਸਟੋਰ ਨਹੀਂ ਕੀਤੀ ਜਾਂਦੀ। ਉਦਾਹਰਣ ਵਜੋਂ, ਏਜੰਟ ਕਈ-ਦੌਰ ਦੀ ਯੋਜਨਾਬੰਦੀ ਗੱਲਬਾਤ ਦੌਰਾਨ ਤੱਥ ਇਕਠੇ ਕਰ ਸਕਦਾ ਹੈ ਅਤੇ ਉਹਨਾਂ ਨੂੰ ਆਖਰੀ ਯਾਤਰਾ ਕਾਰਡ ਤਿਆਰ ਕਰਨ ਲਈ ਵਰਤ ਸਕਦਾ ਹੈ।

### ਲਾਂਗ-ਟਾਰਮ ਮੈਮੋਰੀ
ਪਸੰਦਾਂ ਅਤੇ ਤੱਥ ਜੋ **ਸੈਸ਼ਨਾਂ ਵਿੱਚ ਲਗਾਤਾਰ** ਟਿਕ ਕੇ ਰਹਿੰਦੇ ਹਨ। ਵਾਪਸ ਆਉਣ ਵਾਲੇ ਯੂਜ਼ਰ ਨੂੰ ਆਪਣੀਆਂ ਖੁਰਾਕ ਦੀਆਂ ਰੋਕਥਾਮਾਂ ਜਾਂ ਯਾਤਰਾ ਸਟਾਈਲ ਦੁਬਾਰਾ ਦੱਸਣ ਦੀ ਲੋੜ ਨਹੀਂ ਹੋਣੀ ਚਾਹੀਦੀ। ਲਾਂਗ-ਟਾਰਮ ਮੈਮੋਰੀ ਆਮ ਤੌਰ 'ਤੇ ਬਾਹਰੀ ਸਟੋਰ — ਡੇਟਾਬੇਸ, ਫਾਈਲ, ਜਾਂ ਵੇਕਟਰ ਇੰਡੈਕਸ — ਦੁਆਰਾ ਸਮਰਥਿਤ ਹੁੰਦੀ ਹੈ ਅਤੇ ਟੂਲਾਂ ਰਾਹੀਂ ਏਜੰਟ ਨੂੰ ਉਪਲੱਬਧ ਕਰਵਾਈ ਜਾਂਦੀ ਹੈ।


## ਵਰਕਿੰਗ ਮੈਮਰੀ ਸੈਸ਼ਨਾਂ ਨਾਲ

ਸਭ ਤੋਂ ਸਧਾਰਣ ਸਰੂਪ ਦੀ ਮੈਮਰੀ ਗੱਲਬਾਤ ਸੈਸ਼ਨ ਹੁੰਦੀ ਹੈ। ਜਦੋਂ ਤੁਸੀਂ ਇਕੋ ਹੀ ਸੈਸ਼ਨ (`agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਨੂੰ ਲਗਾਤਾਰ `agent.run()` ਕਾਲਾਂ ਵਿੱਚ ਦਿੰਦੇ ਹੋ, ਤਾਂ ਏਜੰਟ ਉਸ ਗੱਲਬਾਤ ਦਾ ਪੂਰਾ ਇਤਿਹਾਸ ਦੇਖਦਾ ਹੈ ਅਤੇ ਪਹਿਲਾਂ ਦੀਆਂ ਜਾਣਕਾਰੀਆਂ ਯਾਦ ਕਰ ਸਕਦਾ ਹੈ।

ਆਓ ਇੱਕ ਟਰੇਵਲ ਏਜੰਟ ਬਣਾਈਏ ਅਤੇ ਵਰਕਿੰਗ ਮੈਮਰੀ ਨੂੰ ਦਰਸਾਈਏ।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ਏਜੰਟ ਨੇ ਬਜਟ ਨੂੰ ਸਹੀ ਤਰੀਕੇ ਨਾਲ ਯਾਦ ਕੀਤਾ ਕਿਉਂਕਿ ਦੋਹਾਂ ਸੁਨੇਹਿਆਂ ਦਾ ਇੱਕੋ ਹੀ ਸੈਸ਼ਨ ਸੀ। ਇਹ **ਵਰਕਿੰਗ ਮੈਮੋਰੀ** ਹੈ — ਇਹ ਸਿਰਫ਼ ਸੈਸ਼ਨ ਦੇ ਜੀਵਨਕਾਲ ਲਈ ਮੌਜੂਦ ਰਹਿੰਦੀ ਹੈ।

### ਇੱਕ ਨਵੀਂ ਥ੍ਰੇਡ ਨਾਲ ਕੀ ਹੁੰਦਾ ਹੈ?

ਜੇ ਅਸੀਂ ਇੱਕ **ਨਵਾਂ** ਸੈਸ਼ਨ ਬਣਾਉਂਦੇ ਹਾਂ, ਤਾਂ ਏਜੰਟ ਨੂੰ ਪਿਛਲੇ ਗੱਲਬਾਤ ਦੀ ਕੋਈ ਯਾਦ ਨਹੀਂ ਰਹਿੰਦੀ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ਲੰਬੇ ਸਮੇਂ ਦੀ ਯਾਦ ਮਿਸਾਲ

ਯੂਜ਼ਰ ਦੀਆਂ ਪਸੰਦਾਂ ਨੂੰ **ਸੈਸ਼ਨਾਂ ਦੇ ਅੰਦਰੂਨੀ ਵਾਰ-ਵਾਰ ਯਾਦ ਰੱਖਣ ਲਈ**, ਸਾਨੂੰ ਇੱਕ ਅਜਿਹਾ ਸਥਾਈ ਸਟੋਰ ਚਾਹੀਦਾ ਹੈ ਜੋ ਗੱਲਬਾਤ ਦੀ ਧਾਗੇ ਤੋਂ ਬਾਹਰ ਰਹਿੰਦਾ ਹੈ। ਏਜੰਟ ਇਸ ਸਟੋਰ ਤੱਕ **ਟੂਲਜ਼** ਰਾਹੀਂ ਪਹੁੰਚ ਕਰਦਾ ਹੈ — ਫੰਕਸ਼ਨਾਂ ਦ੍ਵਾਰਾ ਜੋ ਇਹ ਜਾਣਕਾਰੀ ਸੰਭਾਲਣ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਸਧਾਰਣ ਇਨ-ਮੈਮੋਰੀ ਪਸੰਦ ਸਟੋਰ ਨੂੰ ਲਾਗੂ ਕਰਦੇ ਹਾਂ (ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ ਇਸ ਦੇ ਲਈ ਡੇਟਾਬੇਸ ਜਾਂ ਵੈਕਟਰ ਇੰਡੈਕਸ ਦਾ ਸਹਾਰਾ ਲਵੋਗੇ) ਅਤੇ ਇਸਨੂੰ ਐਜੰਟ ਵੱਲੋਂ ਵਰਤਣ ਵਾਲੇ ਟੂਲਜ਼ ਵਜੋਂ ਪ੍ਰਦਰਸ਼ਿਤ ਕਰਦੇ ਹਾਂ।

### ਬਣਤਰ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — ਪਹਿਲੀ ਵਾਰੀ ਯੂਜ਼ਰ ਸਾਲਗਿਰਾਹ ਦੀ ਯਾਤਰਾ ਬੁੱਕ ਕਰਦਾ ਹੈ

ਸਾਰਾਹ ਪਹਿਲੀ ਵਾਰੀ ਆਉਂਦੀ ਹੈ। ਏਜੰਟ ਨੂੰ ਉਨ੍ਹਾਂ ਦੀਆਂ ਪਸੰਦਾਂ ਟੂਲਜ਼ ਰਾਹੀਂ ਸੁਰੱਖਿਅਤ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ ਅਤੇ ਉਨ੍ਹਾਂ ਨੂੰ ਹੋਟਲ ਸਿਫਾਰਸ਼ ਕਰਨ ਲਈ ਇਸਤਮਾਲ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ਸੰਜੋਗ 2 — ਸਾਰਾ ਹਫ਼ਤੇ ਬਾਅਦ ਵਾਪਸ ਆਉਂਦੀ ਹੈ

ਸਾਰਾ ਇੱਕ **ਨਵੀਂ ਥਰੇਡ** ਸ਼ੁਰੂ ਕਰਦੀ ਹੈ (ਨਵੀਂ ਸੈਸ਼ਨ ਦੀ ਨਕਲ ਕਰਦੀ ਹੈ)। ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਖਾਲੀ ਹੈ, ਪਰ ਲੰਬੇ ਸਮੇਂ ਦੀ ਪਸੰਦ ਸਟੋਰ ਵਿੱਚ ਅਜੇ ਵੀ ਉਸਦੀ ਜਾਣਕਾਰੀ ਮੌਜੂਦ ਹੈ। ਏਜੰਟ ਨੂੰ ਇਸਨੂੰ ਪ੍ਰਾਪਤ ਕਰਕੇ ਸਿਫਾਰਸ਼ਾਂ ਨੂੰ ਵਿਅਕਤਿਗਤ ਬਣਾਉਣ ਲਈ ਵਰਤਣਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਤਿੰਨ ਕਿਸਮਾਂ ਦੀ ਏਜੰਟ ਮੈਮੋਰੀ ਬਾਰੇ ਸਿੱਖਿਆ ਅਤੇ ਇਹਨਾਂ ਨੂੰ Microsoft Agent Framework ਨਾਲ ਕਿਵੇਂ ਲਾਗੂ ਕਰਨਾ ਹੈ:

| ਮੈਮੋਰੀ ਦੀ ਕਿਸਮ | MAF ਮੈਕੈਨਿਜ਼ਮ | ਜੀਵਨਕਾਲ |
|---|---|---|
| **ਕੰਮ ਕਰਨ ਵਾਲੀ** | `agent.create_session()` | ਇੱਕ ਹੀ ਗੱਲਬਾਤ |
| **ਛੋਟੀ ਮਿਆਦ ਦੀ** | ਇੱਕ ਥਰੇਡ ਵਿੱਚ ਸੰਗ੍ਰਹਿਤ ਸੰਦਰਭ | ਇੱਕ ہی ਕੰਮ / ਸੈਸ਼ਨ |
| **ਲੰਮੀ ਮਿਆਦ ਦੀ** | `@tool` ਫੰਕਸ਼ਨਾਂ ਰਾਹੀਂ ਭਰਤੀ ਬਾਹਰੀ ਸਟੋਰ | ਸੈਸ਼ਨਾਂ ਵਿਚਕਾਰ |

### ਮੁੱਖ ਬਿੰਦੂ
1. **`agent.create_session()`** ਕੰਮ ਕਰਨ ਵਾਲੀ ਮੈਮੋਰੀ ਦਿੰਦਾ ਹੈ — ਏਜੰਟ ਇੱਕ ਸੈਸ਼ਨ ਦੌਰਾਨ ਪੂਰੀ ਗੱਲਬਾਤ ਦਾ ਇਤਿਹਾਸ ਵੇਖਦਾ ਹੈ।
2. **ਨਵੇਂ ਸੈਸ਼ਨ ਸੰਦਰਭ ਖੋਦੇ ਹਨ** — ਬਿਨਾਂ ਲੰਮੀ ਮਿਆਦ ਵਾਲੀ ਮੈਮੋਰੀ ਦੇ ਏਜੰਟ ਪਿਛਲੀਆਂ ਗੱਲਬਾਤਾਂ ਨੂੰ ਯਾਦ ਨਹੀਂ ਰੱਖ ਸਕਦਾ।
3. **`@tool` ਫੰਕਸ਼ਨ** ਇਹ ਖਾਲੀਗੁਜ਼ਾਰੀ ਪੂਰੀ ਕਰਦੇ ਹਨ — ਇਹ ਏਜੰਟ ਨੂੰ ਜਾਣਕਾਰੀ ਸੰਜੋਣ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਦੀ ਆਗਿਆ ਦਿੰਦੇ ਹਨ।
4. **ਵਿਅਕਤੀਗਤ ਬਣਾਉ ਬਿਹਤਰ ਹੁੰਦੀ ਹੈ** — ਜਿੰਨੀ ਵੱਧ ਪਸੰਦਾਂ ਸੰਭਾਲੀਆਂ ਜਾਂਦੀਆਂ ਹਨ, ਐਜੰਟ ਦੇ ਸੁਝਾਅ ਉਹਨਾਂ ਤੋਂ ਵੀ ਵਧੀਆ ਬਣਦੇ ਹਨ।

### ਅਸਲ-ਜੀਵਨ ਵਿੱਚ ਲਾਗੂ
- **ਗ੍ਰਾਹਕ ਸੇਵਾ**: ਗ੍ਰਾਹਕਾਂ ਦਾ ਇਤਿਹਾਸ ਅਤੇ ਪਸੰਦਾਂ ਯਾਦ ਰੱਖੋ
- **ਪੈਰਸਨਲ ਅਸਿਸਟੈਂਟਸ**: ਦਿਨਾਂ ਜਾਂ ਹਫ਼ਤਿਆਂ ਦੌਰਾਨ ਸੰਦਰਭ ਬਣਾਈ ਰੱਖੋ
- **ਹੈਲਥਕੇਅਰ**: ਮਰੀਜ਼ ਦੀ ਜਾਣਕਾਰੀ ਅਤੇ ਪਸੰਦਾਂ ਨੂੰ ਟਰੈੱਕ ਕਰੋ
- **ਈ-ਕਾਮਰਸ**: ਇਤਿਹਾਸ ਦੇ ਅਧਾਰ 'ਤੇ ਵਿਅਕਤੀਗਤ ਖਰੀਦਦਾਰੀ

### ਅਗਲੇ ਕਦਮ
- ਇਨ-ਮੇਮੋਰੀ dict ਦੀ ਥਾਂ ਡੇਟਾਬੇਸ ਜਾਂ ਵੇਕਟਰ ਸਟੋਰ ਵਰਗਾ ਕੁਝ (ਜਿਵੇਂ Azure AI Search) ਵਰਤੋਂ
- ਸਮੇਂ-ਸੰਵੇਦਨਸ਼ੀਲ ਜਾਣਕਾਰੀ ਲਈ ਮੈਮੋਰੀ ਦੀ ਮਿਆਦ ਖਤਮ ਹੋਣ ਜੋੜੋ
- ਸਾਂਝੀ ਮੈਮੋਰੀ ਨਾਲ ਬਹੁ-ਏਜੰਟ ਸਿਸਟਮ ਬਣਾਓ
- Cognee ਨੋਟਬੁੱਕ ਵਿੱਚ ਗਿਆਨ-ਗ੍ਰਾਫ-ਆਧਾਰਿਤ ਮੈਮੋਰੀ ਦਾ ਅਧਿਐਨ ਕਰੋ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰਨ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦਿਤ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਦਿਓ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜ਼ਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਵਿਸ਼ੇਸ਼ਜ्ञ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਵਰਣ ਲਈ ਅਸੀਂ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
